> **Legacy exploratory notebook (June 2025, first phase of the project).** Hyper-parameters were chosen on the test files and windows were split at random, so the scores it printed are not trustworthy estimates. It is kept for the record and is audited in `scripts/audit/` and `docs/legacy_audit.html`. Outputs were cleared (they contained participant file names); set `EEGBCI_LEGACY_DATA_DIR` for the helper `utils.py`.

In [ ]:
import sys
sys.path.insert(0, '../')

In [ ]:
import numpy as np
from scipy import stats
from utils import read_eeg, balance_labels, make_features, apply_fbcsp, applyICA, applyPCA

# Data loading

In [ ]:
X_train, y_train = read_eeg(why="train")
print("Before balancing:")
print(np.unique(y_train, return_counts=True))
print("")
X_train, y_train = balance_labels(X_train, y_train)
print("After balancing:")
print(np.unique(y_train, return_counts=True))


In [ ]:
X_train.shape

In [ ]:
X_test = []
y_test = []

for who in range(26):
    X, y = read_eeg(why="test", who=who, verbose=False)
    X = apply_fbcsp(X, y, verbose=False)
    X = applyICA(X, y, n_components=2, verbose=False)
    X = applyPCA(X, y, n_components=2, verbose=False)
    X_test.append(np.array(X))
    y_test.append(np.array(y))

X_test = np.concatenate(X_test, axis=0)
y_test = np.concatenate(y_test, axis=0)


In [ ]:
X_train = apply_fbcsp(X_train, y_train, verbose=False)
X_train = applyICA(X_train, y_train, n_components=2, verbose=False)
X_train = applyPCA(X_train, y_train, n_components=2, verbose=False)


# SVM

In [ ]:
# X_test = np.array(X_test)
# y_test = np.array(y_test)

print(X_test.shape)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
for c in [1, 10, 100]:
    for g in [0.01, 0.1, 1, "scale"]:
        model = SVC(kernel="rbf", C=c, gamma=g)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        print(f"C={c}, gamma={g}: {accuracy_score(y_test, y_pred)}")


In [ ]:
# 📦 Simple MRFO implementation
import numpy as np
from typing import Callable, Tuple, List

IT = 0

def MRFO(
    obj_func: Callable[[np.ndarray], float],
    lb: np.ndarray,
    ub: np.ndarray,
    dim: int,
    pop: int = 30,
    max_iter: int = 100,
    verbose: bool = False,
) -> Tuple[np.ndarray, float, List[float]]:
    global IT
    """
    Manta Ray Foraging Optimization (simplified).

    Parameters
    ----------
    obj_func : callable
        Fitness function to minimise.
    lb, ub   : np.ndarray
        Lower / upper bounds (shape: [dim]).
    dim      : int
        Dimensionality of the search space.
    pop      : int
        Population size.
    max_iter : int
        Number of iterations.
    verbose  : bool
        If True, print best fitness each iteration.

    Returns
    -------
    best_pos      : np.ndarray
        Best solution found.
    best_score    : float
        Objective value of best_pos (minimised).
    fitness_curve : list[float]
        Best score after each iteration.
    """
    if verbose:
        print("==== MRFO ====")
    # ---------- helpers ------------------------------------------------------
    def _bound(x: np.ndarray) -> np.ndarray:
        return np.clip(x, lb, ub)

    # ---------- initialise ---------------------------------------------------
    P = lb + (ub - lb) * np.random.rand(pop, dim)
    # print("1")
    fitness = np.array([obj_func(ind) for ind in P])
    IT = 0
    # print(2)
    best_idx = np.argmin(fitness)
    # print(3)
    best_pos, best_score = P[best_idx].copy(), fitness[best_idx]
    print(f"Best score for now: {-best_score}")
    # print(4)
    curve = [1-best_score]

    # ---------- main loop ----------------------------------------------------
    for t in range(1, max_iter + 1):
        beta = 2 * np.exp(-((t) / max_iter))  # exploitation ↔ exploration

        for i in range(pop):
            r = np.random.rand()
            if r < 0.5:  # -------- Chain foraging
                leader = P[np.random.randint(pop)]
                P[i] = P[i] + np.random.rand() * (leader - P[i])
            else:        # -------- Cyclone foraging
                A = 2 * np.random.rand() * np.sin(2 * np.pi * np.random.rand())
                P[i] = best_pos + A * (best_pos - P[i]) * beta

            # -------- Somersault foraging (global dance)
            S = 2 * np.random.rand() * best_pos - np.random.rand() * P[i]
            P[i] = P[i] + S * beta

            P[i] = _bound(P[i])

        # -------- evaluate ---------------------------------------------------
        fitness = np.array([obj_func(ind) for ind in P])
        IT = 0
        min_idx = np.argmin(fitness)
        if fitness[min_idx] < best_score:
            best_score = fitness[min_idx]
            best_pos = P[min_idx].copy()
        print(f"Best score for now: {-best_score}")

        curve.append(-best_score)
        if verbose:
            print(f"Iter {t:03}/{max_iter}  |  best = {(-best_score):.6f}")

    return best_pos, best_score, curve

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 🔍 Cell N –– Hyper-parameter search with MRFO
# Assumes:  X_train, y_train already prepared and scaled
#           MRFO() function / class already available in the notebook

import numpy as np
from sklearn.model_selection import cross_val_score

def fitness_fn(position):
    C, gamma = position
    model = SVC(C=C, gamma=gamma, kernel='rbf')
    model.fit(X_train, y_train)
    
    testD_2d = X_test.reshape(1, -1) if X_test.ndim == 1 else X_test
    y_pred = model.predict(testD_2d)
    accuracy = accuracy_score(y_test, y_pred)
    return -accuracy  # Minimize the negative accuracy


# --- MRFO search space -------------------------------------------------------
lb = np.array([1e-3, 1e-4])   # lower bounds:  C, gamma
ub = np.array([1000,  1.0 ])   # upper bounds:  C, gamma
dim = 2

# --- Launch optimisation -----------------------------------------------------
best_pos, best_score, curve = MRFO(fitness_fn,
                               lb=lb, ub=ub, dim=dim,
                               pop=20, max_iter=50, verbose=True)

best_C, best_gamma = best_pos
print(f"Best C     : {best_C:.6f}")
print(f"Best gamma : {best_gamma:.6f}")
print(f"CV accuracy: {-best_score:.4f}")

 best accuracy: 44.3%